- Process:
- Load Model, Load Data
- Load Model into TransformerLens, and setup hooks
- Setup genearlized function for viewing atteniton heads.

In [53]:
from pathlib import Path

MODEL_DIR  = Path("model/BD_llama_6heads_1epoch_4layers")
DATA_DIR   = Path("data/BD_llama_inital")
REMAP_PATH = DATA_DIR / "old_to_new.json"
TOKENS_PATH = DATA_DIR / "bios_postreduce.bin"

### Tokenizer

`CondensedTokenizer` wraps the GPT-2 tokenizer + the `old_to_new` remap saved by `main.py`. It encodes text → reduced ids and decodes back.

In [54]:
from condensed_tokenizer import CondensedTokenizer
from bio_sampler import BioSampler

tokenizer = CondensedTokenizer.from_remap_path(REMAP_PATH)
sampler   = BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=0)

print(f"vocab_size = {tokenizer.vocab_size}, eos_token_id = {tokenizer.eos_token_id}")
print(f"{len(sampler.people):,} people, {sampler.n_templates} templates/person\n")

# Specific person + specific template
text = sampler.render(sampler.people[0], exposure_idx=0)
print("render(people[0], 0) →", repr(text))
print("encode →", tokenizer.encode(text)[:15], "...\n")

# Random bio
draw = sampler.sample()
print(f"sample() → person id={draw['person']['id']}, template={draw['exposure_idx']}")
print("text →", repr(draw["text"]))

vocab_size = 1836, eos_token_id = 1835
50,000 people, 46 templates/person

render(people[0], 0) → ' Gabriella Ella Rigby was born on February 18, 1816.'
encode → [870, 83, 882, 663, 5, 1273, 267, 80, 536, 52, 487, 237, 1, 237, 256] ...

sample() → person id=50494, template=26
text → ' Marco Jackson Rowland arrived in this world on December 24, 1717, a day to be remembered.'


### Load model into TransformerLens

The HF checkpoint is a `LlamaForCausalLM` with reduced vocab. We wrap it in `HookedTransformer` so we get hooks + `run_with_cache`.

The `"meta-llama/Llama-2-7b-hf"` string is just a *template* TransformerLens uses to pick the Llama weight-converter — when `hf_model` is provided, every dim and every weight is read from that object. Pre/post-processing flags are off (`fold_ln`, `center_writing_weights`, `center_unembed`) so the loaded weights stay mathematically identical to the HF checkpoint; flip them on later if you want the canonical interp-friendly form.

In [55]:
import torch
from transformers import LlamaForCausalLM
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_llama_weights


def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = pick_device()
dtype = torch.float32

hf_model = LlamaForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=dtype)
hf_model.eval()
assert hf_model.config.vocab_size == tokenizer.vocab_size, (
    f"checkpoint vocab {hf_model.config.vocab_size} != remap vocab "
    f"{tokenizer.vocab_size} — wrong old_to_new.json for this model."
)

# Build the TL config from the HF config so dims match our custom
# 4-layer / hidden=384 / vocab=1836 model (from_pretrained would have used
# the Llama-2-7b template config and tried to read layer 4 of a 4-layer model).
hf_cfg = hf_model.config
tl_cfg = HookedTransformerConfig(
    n_layers = hf_cfg.num_hidden_layers,
    d_model = hf_cfg.hidden_size,
    d_head = hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    n_heads = hf_cfg.num_attention_heads,
    d_mlp= hf_cfg.intermediate_size,
    d_vocab=hf_cfg.vocab_size,
    n_ctx=hf_cfg.max_position_embeddings,
    act_fn="silu",
    normalization_type="RMS",
    gated_mlp=True,
    positional_embedding_type="rotary",
    rotary_base=int(getattr(hf_cfg, "rope_theta", 10000.0)),
    rotary_dim=hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    final_rms=True,
    tie_word_embeddings=hf_cfg.tie_word_embeddings,
    initializer_range=hf_cfg.initializer_range,
    n_key_value_heads=hf_cfg.num_key_value_heads,
    device=device,
)

# Pre-tokenize everything you feed the model; don't attach the tokenizer.
# TL only calls into the tokenizer for `model(str)` / `to_tokens` / `to_string`,
# none of which we use — we always pass token ids directly.
state_dict = convert_llama_weights(hf_model, tl_cfg)
model = HookedTransformer(tl_cfg)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print(f"Loaded on {device}: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}, "
      f"n_heads={model.cfg.n_heads}, d_vocab={model.cfg.d_vocab}")

Loading weights: 100%|██████████| 38/38 [00:00<00:00, 23199.93it/s]

Moving model to device:  mps
Loaded on mps: n_layers=4, d_model=384, n_heads=6, d_vocab=1836


### DataLoader

Iterates `bios_postreduce.bin` as fixed-length chunks of reduced-vocab ids. Same shape the model saw during training.

In [56]:
from data_loader import make_dataloader

loader = make_dataloader(TOKENS_PATH, seq_len=512, batch_size=4, shuffle=False)
print(f"{len(loader.dataset):,} sequences, {len(loader):,} batches")

batch = next(iter(loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("first sequence decoded:")
print(tokenizer.decode(batch["input_ids"][0][:80]))

175,783 sequences, 43,946 batches
{'input_ids': (4, 512), 'labels': (4, 512)}
first sequence decoded:
 Grace Caroline Hancock was brought into existence on April 11, 1711. Veronica Daisy Long celebrates their special day each year on March 10, 1872. Ana Eden Spence rejoices on March 15, 1850, the day they were born. Jennifer Kaitlyn Patrick's birth date is October 13, 1718. Leonardo Xavier Pritchard acknowledges June 23, 1889 as the


In [57]:
def show_tokens(ids, tokenizer, addOne=False):
    """Print each token with its position and decoded text (with repr so
    leading spaces / newlines stay visible).

    addOne=True shifts the displayed index by 1, so positions match what
    the model sees after a BOS/EOS is prepended to `ids` downstream.
    """
    if hasattr(ids, "tolist"):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]   # unwrap [1, N] batch

    offset = 1 if addOne else 0
    id_w = max(len(str(t)) for t in ids)
    idx_w = len(str(len(ids) - 1 + offset))
    print(f"{'idx':>{idx_w}} | {'id':>{id_w}} | text")
    print("-" * (idx_w + id_w + 12))
    for i, t in enumerate(ids):
        print(f"{i + offset:>{idx_w}} | {t:>{id_w}} | {tokenizer.decode([t])!r}")


# Decomposing Activations

Sample Person + Prompt

In [58]:
sample = sampler.sample()
sample

{'person': {'id': 5306,
  'first_name': 'Giovanni',
  'middle_name': 'Weston',
  'last_name': 'Greenwood',
  'birthday': 15,
  'birthmonth': 'October',
  'birthyear': 1815,
  'birthcity': 'Garland, TX',
  'university': 'University of Texas Health Science Center at San Antonio',
  'field': 'Computer Information Systems',
  'company1name': 'AIG',
  'company1city': 'New York, NY'},
 'exposure_idx': 16,
 'text': ' Giovanni Weston Greenwood commemorates their birth anniversary on October 15, 1815.'}

In [59]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device)
show_tokens(tokenizer.encode(sample["text"]), tokenizer, addOne=True)

idx |   id | text
------------------
 1 | 1834 | ' Giovanni'
 2 | 1771 | ' Weston'
 3 | 1694 | ' Greenwood'
 4 | 1377 | ' commemor'
 5 |  153 | 'ates'
 6 |  118 | ' their'
 7 |  495 | ' birth'
 8 |  826 | ' anniversary'
 9 |   52 | ' on'
10 |  426 | ' October'
11 |  242 | ' 15'
12 |    1 | ','
13 |  237 | ' 18'
14 |  241 | '15'
15 |    2 | '.'


Does Our model even get it right?

In [60]:
promptEndIndx = 9

logits = model(input_tokens, return_type="logits")
print("Token Shape: ", input_tokens.shape)
print("Logits Shape: ", logits.shape)


print("Model Prediction (After Prompt):", tokenizer.decode(logits.argmax(-1)[0, promptEndIndx:]))
print("Correct Prediction (After Prompt):", tokenizer.decode(input_tokens[0, promptEndIndx+1:]))

Token Shape:  torch.Size([1, 16])
Logits Shape:  torch.Size([1, 16, 1836])
Model Prediction (After Prompt):  October 15, 1815.
Correct Prediction (After Prompt):  October 15, 1815.


### Activation Maps

In [61]:
logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True)

In [62]:
import circuitsvis as cv
for layer_x in range(hf_cfg.num_hidden_layers):
    print(f"Attention Heads from Layer {layer_x}: ")
    display(
        cv.attention.attention_patterns(
            tokens=[tokenizer.decode([tok]) for tok in input_tokens.squeeze().tolist()],
            attention=cache['pattern', layer_x]
        )
    )

Attention Heads from Layer 0: 


Attention Heads from Layer 1: 


Attention Heads from Layer 2: 


Attention Heads from Layer 3: 


# Logit Lens

In [91]:
sample = sampler.sample()
sample

{'person': {'id': 67013,
  'first_name': 'Veronica',
  'middle_name': 'Autumn',
  'last_name': 'Ashworth',
  'birthday': 6,
  'birthmonth': 'November',
  'birthyear': 1794,
  'birthcity': 'Lubbock, TX',
  'university': 'Vanderbilt University',
  'field': 'Sports Management',
  'company1name': 'NASA',
  'company1city': 'Washington, DC'},
 'exposure_idx': 31,
 'text': ' Veronica Autumn Ashworth acknowledges their birth day as November 6, 1794.'}

In [92]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device)
show_tokens(tokenizer.encode(sample["text"]), tokenizer, addOne=True)

idx |   id | text
------------------
 1 | 1718 | ' Veronica'
 2 | 1484 | ' Autumn'
 3 |  697 | ' Ash'
 4 |  743 | 'worth'
 5 | 1249 | ' acknowledges'
 6 |  118 | ' their'
 7 |  495 | ' birth'
 8 |  216 | ' day'
 9 |   71 | ' as'
10 |  441 | ' November'
11 |  162 | ' 6'
12 |    1 | ','
13 |  273 | ' 17'
14 |  606 | '94'
15 |    2 | '.'


In [93]:
promptEndIndx = 9

logits = model(input_tokens, return_type="logits")
print("Token Shape: ", input_tokens.shape)
print("Logits Shape: ", logits.shape)


print("Model Prediction (After Prompt):", tokenizer.decode(logits.argmax(-1)[0, promptEndIndx:]))
print("Correct Prediction (After Prompt):", tokenizer.decode(input_tokens[0, promptEndIndx+1:]))

Token Shape:  torch.Size([1, 16])
Logits Shape:  torch.Size([1, 16, 1836])
Model Prediction (After Prompt):  November 6, 1794.
Correct Prediction (After Prompt):  November 6, 1794.


In [106]:
logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True)

In [107]:
for layer_idx in range(model.cfg.n_layers):
    intermediateLogits = (cache['resid_pre',layer_idx] @ model.W_U)
    pred = tokenizer.decode(intermediateLogits.argmax(-1)[promptEndIndx:promptEndIndx+1])
    print(f"-----Layer {layer_idx} Prediction: {pred} | Top k Preds")

finalPredictions = (cache['resid_pre',layer_idx] @ model.W_U)
pred = tokenizer.decode(intermediateLogits.argmax(-1)[promptEndIndx:promptEndIndx+1])
print(f"-----Layer {layer_idx} Prediction: {pred} | Top k Preds")


Layer 0 Prediction:  as
Layer 1 Prediction:  September
Layer 2 Prediction:  September
Layer 3 Prediction:  October


In [103]:
model.cfg.n_layers

4